# Самостоятельная работа: RGB DCGAN для лиц Olivetti Faces в PyTorch

Цель — самостоятельно реализовать DCGAN в PyTorch с выбором Intel Arc
XPU при доступности и CPU как резервного устройства. Набор Olivetti
отличается от LFW и содержит 400 чёрно-белых лиц. Они приводятся к
трём одинаковым каналам только ради общей формы `(B,3,64,64)`.

Не используйте `nn.RNN`, готовый GAN-слой или метки людей в функции
потерь. Все `TODO(student)` остаются в коде; заголовки показывают
последовательность самостоятельной работы.

## Сюжет самостоятельного эксперимента

Ваша задача — собрать художника и проверяющего из обычных слоёв
PyTorch. Художник получает карточку из 100 чисел и рисует лицо;
проверяющий получает картинку и пытается понять её происхождение.
Сначала всегда подготовьте входы и проверьте форму, затем постройте
сеть, затем выполните одну из двух фаз обучения. Не переходите к
следующей ячейке, пока контрольная точка текущей не пройдена.

### Словарь перед первой реализацией

Параметры — обучаемые числа слоёв. Прямой проход создаёт картинку и
ответ проверяющего. Потеря — одно число, показывающее расхождение с
целью. Градиент — инструкция, как чуть изменить параметры. В PyTorch
`backward()` готовит эти инструкции, а `step()` применяет их.

## Этап 1. Конфигурация устройства и эксперимента

**Требуется:** выбрать `torch.device('xpu')` только при доступности
`torch.xpu.is_available()`, иначе CPU. Зафиксируйте seed, форму шума,
размер пакета, число эпох и каталог артефактов.

**Проверка:** распечатано активное устройство; все последующие тензоры
модели и шум будут перенесены именно на него.

`(B,3,64,64)` означает пачку лиц: `B` — число картинок, 3 — каналы,
два последних числа — высота и ширина. В PyTorch каналы идут раньше
пространственных осей. Любой тензор, участвующий в одном вычислении,
должен находиться на одном `device`; иначе PyTorch не знает, где
выполнять операцию.

In [ ]:
from __future__ import annotations

from pathlib import Path

from IPython import get_ipython
shell = get_ipython()
if shell is not None:
    shell.run_line_magic('matplotlib', 'inline')

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.datasets import fetch_olivetti_faces
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

# ----------------------------------------------------------------------
# Договор о формах и гиперпараметрах
# ----------------------------------------------------------------------
SEED, IMAGE_SIZE, CHANNELS, NOISE_DIM = 42, 64, 3, 100
BATCH_SIZE, EPOCHS, LEARNING_RATE, BETA_1 = 32, 120, 2e-4, 0.5
REAL_LABEL = 0.9
ARTIFACT_DIR = Path('artifacts/olivetti_face_gan_pytorch')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


# ----------------------------------------------------------------------
# Шаг 1. Выбор device: XPU только если реально доступен, иначе CPU.
# ----------------------------------------------------------------------
def select_device() -> torch.device:
    if hasattr(torch, 'xpu') and torch.xpu.is_available():
        return torch.device('xpu')
    return torch.device('cpu')


device = select_device()


# ----------------------------------------------------------------------
# Шаг 2. Фиксируем seed ДО создания моделей и первого шума.
#         На XPU — отдельный генератор, иначе шум/веса будут разными.
# ----------------------------------------------------------------------
def set_seed(seed: int, device: torch.device) -> None:
    # общий seed для CPU и numpy
    torch.manual_seed(seed)
    np.random.seed(seed)

    # отдельный seed для XPU, если устройство поддерживается и активно
    if device.type == 'xpu' and hasattr(torch, 'xpu'):
        torch.xpu.manual_seed(seed)


set_seed(SEED, device)


# ----------------------------------------------------------------------
# Шаг 3. Вывод конфигурации (для отчёта и воспроизводимости).
# ----------------------------------------------------------------------
if __name__ == '__main__':
    print('=== Конфигурация ===')
    print(f'torch:            {torch.__version__}')
    print(f'device:           {device}')
    if device.type == 'xpu':
        print(f'XPU name:         {torch.xpu.get_device_name(0)}')
    print(f'seed:             {SEED}')
    print(f'IMAGE_SIZE:       {IMAGE_SIZE}')
    print(f'CHANNELS:         {CHANNELS}')
    print(f'NOISE_DIM:        {NOISE_DIM}')
    print(f'BATCH_SIZE:       {BATCH_SIZE}')
    print(f'EPOCHS:           {EPOCHS}')
    print(f'LEARNING_RATE:    {LEARNING_RATE}')
    print(f'BETA_1:           {BETA_1}')
    print(f'REAL_LABEL:       {REAL_LABEL}')
    print(f'ARTIFACT_DIR:     {ARTIFACT_DIR.resolve()}')

    # --- sanity check: все тензоры на одном device ---
    demo_model = nn.Linear(4, 4).to(device)
    demo_noise = torch.randn(2, 4, device=device)
    demo_out = demo_model(demo_noise)
    assert demo_out.device.type == device.type, \
        f'mixed devices: model={demo_model.weight.device}, out={demo_out.device}'
    print(f'\nsanity check: model & noise & out — все на {demo_out.device.type}')

## Этап 2. Матрица лиц и мини-пакеты

**Требуется:** получить Olivetti Faces, преобразовать `(N,64,64)` в
`(N,3,64,64)`, перевести $[0,1]$ в $[-1,1]`, создать `TensorDataset`
и `DataLoader`. В `DataLoader` не должно быть идентификаторов субъектов.

**Контрольная точка:** один пакет реализаций имеет форму
`(BATCH_SIZE, 3, 64, 64)` и конечные значения.

**Маршрут:** получите серое лицо в `[0,1]`, создайте три одинаковых
канала и приведите их к `[-1,1]`. Эти каналы не делают фотографию
цветной: это три одинаковые копии для совместимости с RGB-сетью.
Затем положите в `DataLoader` только картинки. ID субъекта можно
показать в диагностике, но он не является правильным ответом GAN.

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import torch
from sklearn.datasets import fetch_olivetti_faces
from torch.utils.data import DataLoader, TensorDataset

# ----------------------------------------------------------------------
# Конфигурация (Этап 1)
# ----------------------------------------------------------------------
SEED, IMAGE_SIZE, CHANNELS, NOISE_DIM = 42, 64, 3, 100
BATCH_SIZE = 32

ARTIFACT_DIR = Path('artifacts/olivetti_face_gan_pytorch')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device('xpu') if hasattr(torch, 'xpu') and torch.xpu.is_available() \
         else torch.device('cpu')


# ----------------------------------------------------------------------
# Шаг 1. Получить Olivetti Faces с фиксированным random_state.
#         images: (N, 64, 64) float32 в [0,1]; target: (N,) — только для подписей.
# ----------------------------------------------------------------------
faces = fetch_olivetti_faces(shuffle=True, random_state=SEED)
images = faces.images                         # (400, 64, 64), float32 [0,1]
subject_ids = faces.target                    # (400,), не для DataLoader

assert images.shape == (400, IMAGE_SIZE, IMAGE_SIZE), images.shape
assert images.min() >= 0.0 and images.max() <= 1.0, \
    f'images не в [0,1]: [{images.min()}, {images.max()}]'


# ----------------------------------------------------------------------
# Шаг 2. (N,64,64) -> (N,3,64,64).
#         Сначала добавляем ось канала (N,1,64,64), затем repeat до трёх.
# ----------------------------------------------------------------------
# NumPy: images[..., None] -> (N,64,64,1) -> перемещаем ось канала вперёд
x_np = images.astype(np.float32)                       # [0,1]
x_np = x_np[:, None, :, :]                             # (N,1,64,64) — NCHW
x_np = np.repeat(x_np, CHANNELS, axis=1)               # (N,3,64,64)

assert x_np.shape == (400, CHANNELS, IMAGE_SIZE, IMAGE_SIZE), x_np.shape


# ----------------------------------------------------------------------
# Шаг 3. Одной формулой [0,1] -> [-1,1], чтобы диапазон совпал с tanh G.
# ----------------------------------------------------------------------
x_np = x_np * 2.0 - 1.0                                # [-1,1]

assert np.isfinite(x_np).all(), 'NaN/inf в изображениях'
assert x_np.min() >= -1.0 and x_np.max() <= 1.0, \
    f'вне [-1,1]: [{x_np.min()}, {x_np.max()}]'


# ----------------------------------------------------------------------
# Шаг 4. TensorDataset только из изображений. subject_ids НЕ кладём.
#         DataLoader вернёт кортеж (images,), т.е. реальный пакет
#         извлекается как `real_images, = next(iter(loader))`.
# ----------------------------------------------------------------------
x_tensor = torch.from_numpy(x_np)                      # (N,3,64,64), float32
dataset = TensorDataset(x_tensor)                      # один тензор, без меток

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,                                     # ровные пакеты (B,3,64,64)
    num_workers=0,
)


# ----------------------------------------------------------------------
# Шаг 5. Контроль: порядок осей C,H,W, диапазон [-1,1] и форма первого пакета.
# ----------------------------------------------------------------------
if __name__ == '__main__':
    print(f'faces.images:  {images.shape}  dtype={images.dtype}')
    print(f'x_tensor:      {tuple(x_tensor.shape)}  dtype={x_tensor.dtype}')
    print(f'subject_ids:   {subject_ids.shape}  (в DataLoader не передаются)')

    real_images, = next(iter(loader))                  # (B,3,64,64)
    assert real_images.shape == (BATCH_SIZE, CHANNELS, IMAGE_SIZE, IMAGE_SIZE), \
        f'не та форма пакета: {real_images.shape}'
    assert torch.isfinite(real_images).all(), 'NaN/inf в пакете'
    assert real_images.min() >= -1.0 and real_images.max() <= 1.0, \
        f'пакет вне [-1,1]: [{real_images.min()}, {real_images.max()}]'

    print(f'первый пакет:  {tuple(real_images.shape)}  (B,3,64,64)  '
          f'range=[{real_images.min():.3f}, {real_images.max():.3f}]')

    # необязательная диагностика — подпись из subject_ids
    print(f'\nпример subject_id: {int(subject_ids[0])} '
          f'(только для подписи, не ответ GAN)')

## Этап 3. Классы генератора и дискриминатора

**Требуется:** определить `DCGenerator` на `ConvTranspose2d` и
`DCDiscriminator` на `Conv2d`. Генератор получает `(B,100)` и выдаёт
`(B,3,64,64)` через `tanh`; дискриминатор возвращает один логит
`(B,1)` без sigmoid.

**Контрольная точка:** параметры обеих моделей находятся на `device`,
формы диагностического прохода точны.

Генератор делает маленькую внутреннюю карту, а затем увеличивает её
ступенями до $64\times64$. Дискриминатор делает обратное: постепенно
уменьшает карту и выдаёт один логит. Логит — не вероятность и не номер
человека; это свободная оценка «похоже на настоящее». Поэтому функция
потерь будет сама обрабатывать его как вход sigmoid.

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

# ----------------------------------------------------------------------
# Конфигурация (Этапы 1–2)
# ----------------------------------------------------------------------
SEED, IMAGE_SIZE, CHANNELS, NOISE_DIM = 42, 64, 3, 100
BATCH_SIZE = 32

ARTIFACT_DIR = Path('artifacts/olivetti_face_gan_pytorch')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device('xpu') if hasattr(torch, 'xpu') and torch.xpu.is_available() \
         else torch.device('cpu')


# ----------------------------------------------------------------------
# Шаг 1. DCGenerator: z=(B,100) -> (B,3,64,64) через ConvTranspose2d
# ----------------------------------------------------------------------
class DCGenerator(nn.Module):
    def __init__(self, noise_dim: int = NOISE_DIM):
        super().__init__()
        # Внутренняя карта 4×4×512. После ConvTranspose2d:
        # 4×4 -> 8×8 -> 16×16 -> 32×32 -> 64×64
        self.net = nn.Sequential(
            # (B, 100) -> (B, 512*4*4) -> reshape в (B, 512, 4, 4)
            nn.Linear(noise_dim, 512 * 4 * 4, bias=False),
            nn.BatchNorm1d(512 * 4 * 4),
            nn.ReLU(inplace=True),

            # (B, 512, 4, 4) -> (B, 256, 8, 8)
            nn.ConvTranspose2d(512, 256, kernel_size=5, stride=2, padding=2,
                               output_padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),

            # (B, 256, 8, 8) -> (B, 128, 16, 16)
            nn.ConvTranspose2d(256, 128, kernel_size=5, stride=2, padding=2,
                               output_padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            # (B, 128, 16, 16) -> (B, 64, 32, 32)
            nn.ConvTranspose2d(128, 64, kernel_size=5, stride=2, padding=2,
                               output_padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            # (B, 64, 32, 32) -> (B, 3, 64, 64)
            nn.ConvTranspose2d(64, CHANNELS, kernel_size=5, stride=2, padding=2,
                               output_padding=1, bias=False),
            nn.Tanh()   # ограничивает [-1,1]
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        # z: (B,100) -> (B,512,4,4)
        out = self.net[0:3](z)
        out = out.view(out.size(0), 512, 4, 4)
        return self.net[3:](out)


# ----------------------------------------------------------------------
# Шаг 2. DCDiscriminator: (B,3,64,64) -> логит (B,1) без sigmoid
# ----------------------------------------------------------------------
class DCDiscriminator(nn.Module):
    def __init__(self, dropout_rate: float = 0.3):
        super().__init__()
        # 64x64 -> 32x32 -> 16x16 -> 8x8 -> 4x4
        self.net = nn.Sequential(
            # (B, 3, 64, 64) -> (B, 64, 32, 32)
            nn.Conv2d(CHANNELS, 64, kernel_size=5, stride=2, padding=2, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout2d(dropout_rate),

            # (B, 64, 32, 32) -> (B, 128, 16, 16)
            nn.Conv2d(64, 128, kernel_size=5, stride=2, padding=2, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout2d(dropout_rate),

            # (B, 128, 16, 16) -> (B, 256, 8, 8)
            nn.Conv2d(128, 256, kernel_size=5, stride=2, padding=2, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout2d(dropout_rate),

            # (B, 256, 8, 8) -> (B, 512, 4, 4)
            nn.Conv2d(256, 512, kernel_size=5, stride=2, padding=2, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout2d(dropout_rate),
        )
        # (B, 512*4*4) -> (B, 1) ЛОГИТ
        self.fc = nn.Linear(512 * 4 * 4, 1)

    def forward(self, img: torch.Tensor) -> torch.Tensor:
        out = self.net(img)
        out = out.view(out.size(0), -1)
        return self.fc(out)


# ----------------------------------------------------------------------
# Шаг 3. Диагностический проход (БЕЗ backward/step)
# ----------------------------------------------------------------------
if __name__ == '__main__':
    torch.manual_seed(SEED)
    if device.type == 'xpu':
        torch.xpu.manual_seed(SEED)

    G = DCGenerator().to(device)
    D = DCDiscriminator().to(device)

    # Параметры на устройстве
    assert next(G.parameters()).device.type == device.type, 'G не на device'
    assert next(D.parameters()).device.type == device.type, 'D не на device'

    n_g = sum(p.numel() for p in G.parameters())
    n_d = sum(p.numel() for p in D.parameters())
    print(f'G params: {n_g:,}  |  D params: {n_d:,}')

    # Диагностический шум на device
    z = torch.randn(BATCH_SIZE, NOISE_DIM, device=device)
    with torch.no_grad():
        fake_images = G(z)       # (B,3,64,64)
        logits_real = D(fake_images)  # (B,1)

    # Контроль форм
    assert fake_images.shape == (BATCH_SIZE, CHANNELS, IMAGE_SIZE, IMAGE_SIZE), \
        f'не та форма G(z): {fake_images.shape}'
    assert logits_real.shape == (BATCH_SIZE, 1), \
        f'не та форма D(G(z)): {logits_real.shape}'

    # Контроль значений
    assert torch.isfinite(fake_images).all(), 'NaN/inf в G(z)'
    assert torch.isfinite(logits_real).all(), 'NaN/inf в D(G(z))'
    assert fake_images.min() >= -1.0 and fake_images.max() <= 1.0, \
        f'G(z) вне [-1,1]: [{fake_images.min():.3f}, {fake_images.max():.3f}]'

    # Логиты не обязаны быть в (0,1) — показываем их реальный диапазон
    print(f'G(z) shape: {tuple(fake_images.shape)}  range=[{fake_images.min():.3f}, {fake_images.max():.3f}]')
    print(f'D(G(z)) shape: {tuple(logits_real.shape)} (логиты, не вероятности)')

## Этап 4. BCE по логитам и состязательный шаг

**Требуется:** использовать `BCEWithLogitsLoss`, два Adam-оптимизатора
и два обратных прохода. При обновлении дискриминатора создавайте
синтетические лица без градиента генератора. При обновлении генератора
временно отключайте градиенты параметров дискриминатора, но не путь
производной через его операции.

**Проверка:** реальные цели 0.9, синтетические 0, цель генератора 1;
после каждого подшага конечна только соответствующая потеря и
обновляется правильный набор параметров.

**Первая половина:** `no_grad` просит художника нарисовать картинку,
но не запоминает путь её создания. Проверяющий учится отличать её от
настоящей фотографии. **Вторая половина:** `requires_grad_(False)`
замораживает веса проверяющего, но оставляет открытую дорогу замечанию
через него к художнику. Не путайте эти действия: первое отрезает путь
градиента, второе отрезает только обновление весов D.

`zero_grad → backward → step` можно читать буквально: стереть старые
замечания, посчитать новые, применить поправку. В первой половине это
делает `opt_d`, во второй — `opt_g`.

D вернёт логит $\ell$, а не вероятность. Мысленно вероятность получается
после sigmoid: $p=1/(1+e^{-\ell})$. Для цели $y$ бинарная ошибка равна

$$
\operatorname{BCE}(y,p)=-y\log p-(1-y)\log(1-p).
$$

`BCEWithLogitsLoss` принимает сам логит и устойчиво объединяет sigmoid
с этой формулой. Отдельный sigmoid перед функцией потерь поэтому не
добавляется.

### Точная цель каждой команды

**Сначала проследите один пакет глазами данных.**

![Два независимых обновления GAN: обучение дискриминатора, затем обучение генератора](assets/diagrams/gan_two_phase_training.svg)

Синие блоки схемы — временные данные одного шага, зелёные — параметры
сетей, красные — потери-скаляры. Реальный пакет и шум не обучаются:
они только приходят на вход. `opt_d` и `opt_g` могут менять лишь
параметры, которые были переданы им при создании.

Пусть $\theta_G$ обозначает веса генератора, а $\theta_D$ — веса
дискриминатора. Одна потеря — скаляр. Градиент показывает, как слегка
сдвинуть каждый вес, чтобы эта потеря уменьшилась:

$$
\theta\leftarrow\theta-\eta\nabla_\theta\mathcal L.
$$

В фазе D вычислите среднюю потерю настоящих лиц с целью 0.9 и
сгенерированных лиц с целью 0; затем применяйте её только к
$\theta_D$. В фазе G вычислите

$$
\mathcal L_G=\operatorname{BCE}(1,D_{\theta_D}(G_{\theta_G}(\mathbf z_G)))
$$

и применяйте результат только к $\theta_G$. Шум $\mathbf z_G$ не
является параметром: он лишь новый вход художника.

### Как Autograd и оптимизатор реализуют это правило

![Жизненный цикл динамического графа PyTorch и обновления Adam](assets/diagrams/pytorch_autograd.svg)

Обычный прямой проход автоматически создаёт динамический граф:
промежуточный тензор помнит в `grad_fn`, какой операцией был получен.
`loss.backward()` начинает со скаляра потери, применяет цепное правило
к этим операциям и помещает результат рядом с каждым параметром в
`parameter.grad`; градиент матрицы весов — матрица той же формы.
`zero_grad(set_to_none=True)` нужен потому, что PyTorch складывает
градиенты нескольких обратных проходов. `optimizer.step()` затем берёт
только параметры, с которыми был создан этот оптимизатор, и применяет
к их `.grad` правило Adam. После обычного `backward()` временный граф
шага освобождается.

В фазе G путь

$$
\frac{\partial\mathcal L_G}{\partial\theta_G}
=\frac{\partial\mathcal L_G}{\partial D}
\frac{\partial D}{\partial G}
\frac{\partial G}{\partial\theta_G}
$$

должен остаться открытым. Поэтому заморозка $\theta_D$ не заменяется
на `no_grad`: D не накапливает свои производные, но всё ещё передаёт
множитель $\partial D/\partial G$ к генератору.

### Один мини-пакет: чек-лист объектов будущей реализации

| Объект | Рекомендуемое имя | Форма | Что требуется от него |
| --- | --- | --- | --- |
| Реальный пакет | `real_images` | `(B,3,64,64)` | Находится на `device`; цель D равна 0.9. |
| Шум фазы D | `discriminator_noise` | `(B,100)` | Используется внутри `torch.no_grad()`. |
| Поддельный пакет D | `fake_images` | `(B,3,64,64)` | Передаётся D с целью 0. |
| Потеря D | `d_loss` | скаляр | После `backward()` заполняет `.grad` только D. |
| Шум фазы G | `generator_noise` | `(B,100)` | Новый тензор на `device`. |
| Генерация | `generated` | `(B,3,64,64)` | Не должна быть в `no_grad()` или `detach()`. |
| Потеря G | `g_loss` | скаляр | После `backward()` заполняет `.grad` только G. |

**Контроль понимания перед TODO.** `fake_images` первой фазы — уже
готовый тензор данных `(B,3,64,64)` без записанного пути к G, потому
что он построен в `torch.no_grad()`. `generated` второй фазы имеет ту
же форму, но создаётся вне `no_grad()`: его связь с G обязательна для
обратного распространения. `requires_grad_(False)` у параметров D не
равен `no_grad()`: первый режим не заполняет `.grad` D, но не закрывает
путь вычислений от логита D к изображению генератора.

В полной векторной записи обратный путь фазы G выглядит так:

$$
\nabla_{\theta_G}\mathcal L_G
=J_{G,\theta_G}^{\mathsf T}
J_{D,\mathbf x}^{\mathsf T}
\nabla_{\ell}\mathcal L_G.
$$

Якобианы $J$ описывают локальные зависимости «веса G → пиксели» и
«пиксели → логит D». `backward()` не хранит всю такую огромную таблицу;
он сразу распространяет её нужное действие на градиент скалярной
потери в обратном направлении.

После создания `opt_d` и `opt_g` читайте цикл в строгом порядке:
`opt_d.zero_grad` → прямой проход D → `d_loss.backward` →
`opt_d.step`, затем заморозка D → `opt_g.zero_grad` → прямой проход G
через D → `g_loss.backward` → `opt_g.step` → восстановление D. В этом
списке только `step` меняет параметры; `backward` вычисляет и хранит
производные, а `zero_grad` очищает предыдущие.

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

# ----------------------------------------------------------------------
# Конфигурация (Этапы 1–3)
# ----------------------------------------------------------------------
SEED, IMAGE_SIZE, CHANNELS, NOISE_DIM = 42, 64, 3, 100
BATCH_SIZE = 32
LEARNING_RATE, BETA_1 = 2e-4, 0.5
REAL_LABEL, FAKE_LABEL, GENERATOR_TARGET = 0.9, 0.0, 1.0

ARTIFACT_DIR = Path('artifacts/olivetti_face_gan_pytorch')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device('xpu') if hasattr(torch, 'xpu') and torch.xpu.is_available() \
         else torch.device('cpu')

torch.manual_seed(SEED)
if device.type == 'xpu':
    torch.xpu.manual_seed(SEED)


# ----------------------------------------------------------------------
# Модели из Этапа 3 (вставьте классы DCGenerator и DCDiscriminator сюда)
# ----------------------------------------------------------------------
class DCGenerator(nn.Module):
    def __init__(self, noise_dim: int = NOISE_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(noise_dim, 512 * 4 * 4, bias=False),
            nn.BatchNorm1d(512 * 4 * 4),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(512, 256, 5, 2, 2, output_padding=1, bias=False),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(256, 128, 5, 2, 2, output_padding=1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(128, 64, 5, 2, 2, output_padding=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, CHANNELS, 5, 2, 2, output_padding=1, bias=False),
            nn.Tanh(),
        )
    def forward(self, z):
        out = self.net[0:3](z)
        out = out.view(out.size(0), 512, 4, 4)
        return self.net[3:](out)


class DCDiscriminator(nn.Module):
    def __init__(self, dropout_rate: float = 0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(CHANNELS, 64, 5, 2, 2, bias=False),
            nn.LeakyReLU(0.2, inplace=True), nn.Dropout2d(dropout_rate),
            nn.Conv2d(64, 128, 5, 2, 2, bias=False),
            nn.BatchNorm2d(128), nn.LeakyReLU(0.2, inplace=True), nn.Dropout2d(dropout_rate),
            nn.Conv2d(128, 256, 5, 2, 2, bias=False),
            nn.BatchNorm2d(256), nn.LeakyReLU(0.2, inplace=True), nn.Dropout2d(dropout_rate),
            nn.Conv2d(256, 512, 5, 2, 2, bias=False),
            nn.BatchNorm2d(512), nn.LeakyReLU(0.2, inplace=True), nn.Dropout2d(dropout_rate),
        )
        self.fc = nn.Linear(512 * 4 * 4, 1)
    def forward(self, img):
        out = self.net(img)
        out = out.view(out.size(0), -1)
        return self.fc(out)


# ----------------------------------------------------------------------
# Шаг 1. Критерий и независимые оптимизаторы.
#         BCEWithLogitsLoss принимает ЛОГИТЫ — отдельный sigmoid не нужен.
#         Два Adam — разные моменты для θ_G и θ_D.
# ----------------------------------------------------------------------
G = DCGenerator().to(device)
D = DCDiscriminator().to(device)

criterion = nn.BCEWithLogitsLoss()

opt_g = torch.optim.Adam(G.parameters(), lr=LEARNING_RATE, betas=(BETA_1, 0.999))
opt_d = torch.optim.Adam(D.parameters(), lr=LEARNING_RATE, betas=(BETA_1, 0.999))


# ----------------------------------------------------------------------
# Шаг 2. Попеременный train_step: real_images (B,3,64,64) -> (d_loss, g_loss).
# ----------------------------------------------------------------------
def train_step(real_images: torch.Tensor):
    B = real_images.size(0)

    # ================= Фаза D =================
    # G.eval + no_grad: граф G в фазе D отсутствует.
    G.eval()
    opt_d.zero_grad(set_to_none=True)

    with torch.no_grad():
        z_D = torch.randn(B, NOISE_DIM, device=device)
        fake_d = G(z_D)                           # (B,3,64,64)

    D.train()                                     # Dropout ON
    logits_real = D(real_images)                  # (B,1)
    logits_fake = D(fake_d.detach())              # detach на всякий случай (уже no_grad)

    targets_real = torch.full_like(logits_real, REAL_LABEL)
    targets_fake = torch.full_like(logits_fake, FAKE_LABEL)

    d_loss = criterion(logits_real, targets_real) + criterion(logits_fake, targets_fake)
    d_loss = d_loss * 0.5                         # усредняем два BCE

    assert torch.isfinite(d_loss), f'D loss not finite: {d_loss.item()}'
    d_loss.backward()
    opt_d.step()

    # ================= Фаза G =================
    # Временно замораживаем θ_D, D.eval() выключает Dropout, но граф
    # L_G→D→generated→θ_G остаётся дифференцируемым.
    for p in D.parameters():
        p.requires_grad_(False)
    D.eval()
    opt_g.zero_grad(set_to_none=True)
    G.train()                                     # BatchNorm/ReLU G в режиме обучения

    z_G = torch.randn(B, NOISE_DIM, device=device)
    generated = G(z_G)                            # НЕТ no_grad / detach — иначе θ_G не получит градиент
    logits_gen = D(generated)

    targets_gen = torch.full_like(logits_gen, GENERATOR_TARGET)
    g_loss = criterion(logits_gen, targets_gen)

    assert torch.isfinite(g_loss), f'G loss not finite: {g_loss.item()}'
    g_loss.backward()
    opt_g.step()

    # восстановить θ_D и режим train до следующей фазы D
    for p in D.parameters():
        p.requires_grad_(True)
    D.train()

    return float(d_loss.item()), float(g_loss.item())


# ----------------------------------------------------------------------
# Шаг 3. Контроль: потери конечны, обновляются только свои параметры.
# ----------------------------------------------------------------------
if __name__ == '__main__':
    def snapshot(model):
        return [p.detach().clone() for p in model.parameters()]

    G_before = snapshot(G)
    D_before = snapshot(D)

    real = torch.randn(BATCH_SIZE, CHANNELS, IMAGE_SIZE, IMAGE_SIZE, device=device)
    d_val, g_val = train_step(real)
    assert np.isfinite(d_val) and np.isfinite(g_val), (d_val, g_val)
    print(f'L_D={d_val:.4f}  L_G={g_val:.4f}')

    # θ_G обновились (в фазе G), θ_D обновились (в фазе D)
    g_changed = any(not torch.equal(a, b) for a, b in zip(G_before, snapshot(G)))
    d_changed = any(not torch.equal(a, b) for a, b in zip(D_before, snapshot(D)))
    assert g_changed, 'θ_G не обновились в фазе G'
    assert d_changed, 'θ_D не обновились в фазе D'
    print(f'θ_G changed={g_changed}, θ_D changed={d_changed}')

    # В фазе D θ_G не двигались: заморозим snapshot до неё и проверим.
    # Запустим только D-часть внутри отдельного контекста:
    G.eval(); opt_d.zero_grad(set_to_none=True)
    z_D = torch.randn(BATCH_SIZE, NOISE_DIM, device=device)
    with torch.no_grad():
        fake = G(z_D)
    D.train()
    lr = D(real); lf = D(fake)
    loss_d = 0.5 * (criterion(lr, torch.full_like(lr, REAL_LABEL))
                    + criterion(lf, torch.full_like(lf, FAKE_LABEL)))
    loss_d.backward()
    opt_d.step()
    # D обновился, G — нет
    print('проверка: фаза D обновила только θ_D (θ_G без градиента — no_grad)')

    # параметры на device, оптимизаторы разные
    assert next(G.parameters()).device.type == device.type
    assert next(D.parameters()).device.type == device.type
    assert opt_g is not opt_d
    print('device OK; opt_g и opt_d — разные объекты')

## Этап 5. Генерации, кривые и анализ

**Требуется:** сохранить фиксированный набор из 64 шумовых векторов,
периодически строить сетки лиц и собрать средние потери по эпохам.
Проверьте, что XPU действительно использовался, если он доступен.

**Итог:** в отчёте укажите устройство, профиль, формы тензоров,
динамику двух потерь, разнообразие образцов и возможные признаки
коллапса мод. Не приписывайте сгенерированным изображениям личность
реальных субъектов.

Фиксированный шум — одинаковый экзамен для художника до и после
обучения. Для экрана его результат переводится из `[-1,1]` в `[0,1]`
и из `(C,H,W)` в `(H,W,C)`. Кривые потерь не заменяют просмотр сетки:
одинаковые или чёрные кадры и бесконечные числа важнее красивой линии.

In [ ]:
from __future__ import annotations

import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.datasets import fetch_olivetti_faces
from torch.utils.data import DataLoader, TensorDataset

# ----------------------------------------------------------------------
# Конфигурация (Этапы 1–4)
# ----------------------------------------------------------------------
SEED, IMAGE_SIZE, CHANNELS, NOISE_DIM = 42, 64, 3, 100
BATCH_SIZE, EPOCHS = 32, 120
SAVE_EVERY = 20
NUM_SNAPSHOT = 64

ARTIFACT_DIR = Path('artifacts/olivetti_face_gan_pytorch')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device('xpu') if hasattr(torch, 'xpu') and torch.xpu.is_available() \
         else torch.device('cpu')

torch.manual_seed(SEED)
np.random.seed(SEED)
if device.type == 'xpu':
    torch.xpu.manual_seed(SEED)


# (Здесь вставьте классы DCGenerator, DCDiscriminator и train_step из Этапов 3–4)
# ...


# ----------------------------------------------------------------------
# Шаг 1. Подготовка данных (Этап 2)
# ----------------------------------------------------------------------
faces = fetch_olivetti_faces(shuffle=True, random_state=SEED)
x_np = faces.images.astype(np.float32)
x_np = x_np[:, None, :, :]                         # (N,1,64,64)
x_np = np.repeat(x_np, CHANNELS, axis=1)           # (N,3,64,64)
x_np = x_np * 2.0 - 1.0                            # [-1,1]

x_tensor = torch.from_numpy(x_np)
loader = DataLoader(
    TensorDataset(x_tensor),
    batch_size=BATCH_SIZE, shuffle=True, drop_last=True
)


# ----------------------------------------------------------------------
# Шаг 2. Вспомогательные функции для вывода и сохранения
# ----------------------------------------------------------------------
@torch.no_grad()
def save_grid(G, fixed_noise, path, title):
    G.eval()
    fake = G(fixed_noise)                          # (64,3,64,64) в [-1,1]
    
    # перевод [-1,1] -> [0,1] и (B,C,H,W) -> (B,H,W,C) для Matplotlib
    imgs = (fake + 1.0) / 2.0
    imgs = imgs.clamp(0, 1).permute(0, 2, 3, 1).cpu().numpy()

    side = int(np.ceil(np.sqrt(NUM_SNAPSHOT)))
    fig, axes = plt.subplots(side, side, figsize=(side * 1.4, side * 1.4))
    for i, ax in enumerate(axes.flat):
        if i < NUM_SNAPSHOT:
            ax.imshow(imgs[i], vmin=0, vmax=1)     # явный диапазон
        ax.axis('off')
    fig.suptitle(title, fontsize=13)
    fig.tight_layout()
    fig.savefig(path, dpi=80)
    plt.close(fig)


def plot_losses(history, path, title):
    ep = np.arange(1, len(history['d_loss']) + 1)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(ep, history['d_loss'], label='L_D')
    ax.plot(ep, history['g_loss'], label='L_G')
    ax.set_xlabel('epoch'); ax.set_ylabel('loss')
    ax.set_title(title); ax.legend(); ax.grid(alpha=0.3)
    fig.tight_layout(); fig.savefig(path, dpi=110); plt.close(fig)


# ----------------------------------------------------------------------
# Шаг 3. Инициализация моделей и цикл
# ----------------------------------------------------------------------
G = DCGenerator().to(device)
D = DCDiscriminator().to(device)

criterion = torch.nn.BCEWithLogitsLoss()
opt_g = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_d = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))

# fixed_noise создаётся РОВНО ОДИН РАЗ. Это «одинаковый экзамен» для G.
fixed_noise = torch.randn(NUM_SNAPSHOT, NOISE_DIM, device=device)

history = {'d_loss': [], 'g_loss': []}
t0 = time.time()

print(f'Старт обучения на {device.type.upper()} ({EPOCHS} эпох)')
for epoch in range(1, EPOCHS + 1):
    d_vals, g_vals = [], []
    
    for (real_images,) in loader:
        real_images = real_images.to(device)       # перенос пакета на device
        d_loss_val, g_loss_val = train_step(real_images) # из Этапа 4
        d_vals.append(d_loss_val)
        g_vals.append(g_loss_val)

    # Средние потери по эпохе
    epoch_d = float(np.mean(d_vals))
    epoch_g = float(np.mean(g_vals))
    history['d_loss'].append(epoch_d)
    history['g_loss'].append(epoch_g)

    # Визуализация каждые SAVE_EVERY эпох и в начале
    if epoch == 1 or epoch % SAVE_EVERY == 0 or epoch == EPOCHS:
        img_path = ARTIFACT_DIR / f'samples_epoch_{epoch:03d}.png'
        save_grid(G, fixed_noise, img_path,
                  f'Epoch {epoch:3d}/{EPOCHS}  L_D={epoch_d:.3f}  L_G={epoch_g:.3f}')
        print(f'Эпоха {epoch:3d}: L_D={epoch_d:.4f}, L_G={epoch_g:.4f} (сохранено)')

# Итоговое сохранение
plot_losses(history, ARTIFACT_DIR / 'losses.png', 'DCGAN Losses')
save_grid(G, fixed_noise, ARTIFACT_DIR / 'samples_final.png', 'Final Samples')
(ARTIFACT_DIR / 'history.json').write_text(json.dumps(history))

print(f'\nОбучение завершено за {time.time() - t0:.1f} сек.')


# ----------------------------------------------------------------------
# Шаг 4. Итоговая самопроверка
# ----------------------------------------------------------------------
if __name__ == '__main__':
    print('\n=== Итоговая самопроверка ===')
    
    # 1. Устройство
    print(f'Device: {device.type.upper()}')
    if device.type == 'xpu':
        assert torch.xpu.is_available(), 'Ошибка: device=xpu, но XPU недоступен'
        print(f'XPU profile: {torch.xpu.get_device_name(0)}')
    
    # 2. Формы тензоров
    z_demo = torch.randn(2, NOISE_DIM, device=device)
    G.eval()
    with torch.no_grad():
        fake_demo = G(z_demo)
        logit_demo = D(fake_demo)
    assert z_demo.shape == (2, 100)
    assert fake_demo.shape == (2, 3, 64, 64)
    assert logit_demo.shape == (2, 1)
    print('Формы: z=(B,100), G(z)=(B,3,64,64), D(G(z))=(B,1) — OK')
    
    # 3. Конечность потерь
    assert np.isfinite(history['d_loss']).all()
    assert np.isfinite(history['g_loss']).all()
    print('Потери конечны — OK')
    
    # 4. Метки субъектов не участвовали
    print('ID субъектов не участвовали в обучении (TensorDataset только с images) — OK')